In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
try:
    #invoices = pd.read_excel("TEST-DATA\\JULY\\JULY-INVOICES.xlsx")
    invoices = pd.read_excel("2026-BRANDS\\INITIAL-DATA\\OCTOBER\\OCT-INV-WK-2.xlsx")
except:
    invoices = pd.DataFrame()
    print("Invoice file not found")

In [5]:
invoices.columns

Index(['Date', 'No', 'Invoice No.', 'A/C', 'Customer', 'P', 'S', 'B', '200g',
       'O.P.', '800g', '1500g', 'ST', '600g', 'KATIKATI', 'MANDAZI',
       'DOUGHNUTS', '50-50 SLICED', 'MANDO FAMILY PACK', '4 Square',
       'XTRA LOONG', 'B-BITEZ', 'Qty'],
      dtype='object')

In [6]:
try:
    #credit_notes = pd.read_excel("TEST-DATA\\JULY\\JULY-CREDITS.xlsx")
    credit_notes = pd.read_excel("2026-BRANDS\\INITIAL-DATA\\OCTOBER\\OCT-CN-WK-2.xlsx")
except:
    credit_notes = pd.DataFrame()
    print("Credit note file not found")

In [7]:
numeric_columns = ['P', 'S', 'B', '200g', 'O.P.', '800g', '600g', 'MANDAZI', 'DOUGHNUTS', 'MANDO FAMILY PACK', '4 Square', 'XTRA LOONG', 'B-BITEZ', 'Qty']

In [8]:
# Helper function: Remove only one matching invoice per credit note (not all duplicates)
def remove_one_invoice_per_credit(invoices, credit_notes, numeric_columns):
    invoices = invoices.copy()
    credit_notes = credit_notes.copy()
    invoices['is_credit'] = False
    used_credit_idx = set()

    # Normalize numeric comparison so exact invoice/credit matches are evaluated safely.
    for col in numeric_columns:
        if col in invoices.columns:
            invoices[col] = pd.to_numeric(invoices[col], errors='coerce')
        if col in credit_notes.columns:
            credit_notes[col] = pd.to_numeric(credit_notes[col], errors='coerce')

    for inv_idx, invoice_row in invoices.iterrows():
        ac = invoice_row['A/C']
        credit_notes_ac = credit_notes[(credit_notes['A/C'] == ac) & (~credit_notes.index.isin(used_credit_idx))]
        for cred_idx, credit_row in credit_notes_ac.iterrows():
            match = True
            for col in numeric_columns:
                if col not in invoice_row.index or col not in credit_row.index:
                    match = False
                    break
                inv_val = pd.to_numeric(invoice_row[col], errors='coerce')
                cn_val = pd.to_numeric(credit_row[col], errors='coerce')
                if pd.isna(inv_val) or pd.isna(cn_val):
                    match = False
                    break
                if inv_val != -cn_val:
                    match = False
                    break
            if match:
                invoices.at[inv_idx, 'is_credit'] = True
                used_credit_idx.add(cred_idx)
                break
    return invoices

In [9]:
# Remove only one matching invoice per credit note (not all duplicates)
invoices = remove_one_invoice_per_credit(invoices, credit_notes, numeric_columns)

# Filter out matching invoice rows
filtered_df = invoices[~invoices['is_credit']].copy()
# Drop the helper column
filtered_df = filtered_df.drop(columns=['is_credit'])

# Group by Date, A/C and Customer, summing numeric columns
final_df = filtered_df.groupby(['Date', 'A/C', 'Customer'], as_index=False)[numeric_columns].sum()

In [10]:
invoices.head()

,Date,No,Invoice No.,A/C,Customer,P,S,B,200g,O.P.,...,KATIKATI,MANDAZI,DOUGHNUTS,50-50 SLICED,MANDO FAMILY PACK,4 Square,XTRA LOONG,B-BITEZ,Qty,is_credit
0,2026-10-04,529426021,1210118,5294,NGORIKA SECONDARY SCHOOL- OLKALAU (5294),0,0,0,0,0,...,0,0,0,0,0,230,0,0,230,False
1,2026-10-04,1674260664,1210120,1674,NAIVAS SUPERMARKETS - WESTSIDE NAKURU (1674),0,15,15,0,5,...,0,0,0,0,0,5,0,0,78,False
2,2026-10-04,1824260719,1210121,1824,NAIVAS SUPERMARKET - MIDTOWN NAKURU (1824),0,30,30,0,10,...,0,0,0,0,0,10,0,0,155,False
3,2026-10-04,957260722,1210122,957,NAIVAS SUPERMARKET-NAKURU (0957),0,15,10,0,4,...,0,0,0,0,0,3,0,0,65,False
4,2026-10-04,1578260503,1210123,1578,WOOLMART SUPERMARTKET - NAKURU CENTRAL (1578),0,10,10,20,0,...,0,0,0,0,0,0,7,0,79,False


In [11]:
invoices.tail()

,Date,No,Invoice No.,A/C,Customer,P,S,B,200g,O.P.,...,KATIKATI,MANDAZI,DOUGHNUTS,50-50 SLICED,MANDO FAMILY PACK,4 Square,XTRA LOONG,B-BITEZ,Qty,is_credit
7001,2026-10-10,618926276,1220406,6189,KANDARA SALES (6189),0,110,148,143,0,...,0,0,0,0,0,75,62,0,579,False
7002,2026-10-10,638126411,1220410,6381,MATARA SALES,0,105,60,264,15,...,0,0,0,0,0,30,56,0,530,False
7003,2026-10-10,606226445,1220412,6062,KITHIMANI SALES,0,251,195,96,10,...,0,0,0,0,0,0,133,0,805,False
7004,2026-10-10,606426447,1220420,6064,NGARARIGA SALES,0,59,48,120,27,...,0,0,0,0,0,0,19,0,344,False
7005,2026-10-10,638926247,1221358,6389,ELDORET DEPOT SALES - ETR DEVICE (6389),0,61,72,6,0,...,0,0,0,0,0,0,0,0,307,False


In [12]:
invoices.isna().sum()

Date                 0
No                   0
Invoice No.          0
A/C                  0
Customer             0
P                    0
S                    0
B                    0
200g                 0
O.P.                 0
800g                 0
1500g                0
ST                   0
600g                 0
KATIKATI             0
MANDAZI              0
DOUGHNUTS            0
50-50 SLICED         0
MANDO FAMILY PACK    0
4 Square             0
XTRA LOONG           0
B-BITEZ              0
Qty                  0
is_credit            0
dtype: int64

In [13]:
invoices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7006 entries, 0 to 7005
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Date               7006 non-null   datetime64[ns]
 1   No                 7006 non-null   int64         
 2   Invoice No.        7006 non-null   int64         
 3   A/C                7006 non-null   int64         
 4   Customer           7006 non-null   object        
 5   P                  7006 non-null   int64         
 6   S                  7006 non-null   int64         
 7   B                  7006 non-null   int64         
 8   200g               7006 non-null   int64         
 9   O.P.               7006 non-null   int64         
 10  800g               7006 non-null   int64         
 11  1500g              7006 non-null   int64         
 12  ST                 7006 non-null   int64         
 13  600g               7006 non-null   int64         
 14  KATIKATI

In [14]:
invoices.describe()

,Date,No,Invoice No.,A/C,P,S,B,200g,O.P.,800g,...,600g,KATIKATI,MANDAZI,DOUGHNUTS,50-50 SLICED,MANDO FAMILY PACK,4 Square,XTRA LOONG,B-BITEZ,Qty
count,7006,7.006000e+03,7.006000e+03,7006.000000,7006.000000,7006.000000,7006.000000,7006.000000,7006.000000,7006.000000,...,7006.000000,7006.0,7006.000000,7006.000000,7006.0,7006.000000,7006.000000,7006.000000,7006.000000,7006.000000
mean,2026-10-07 02:02:05.378247168,1.511463e+09,1.214633e+06,2192.321439,3.019412,75.256923,33.783043,86.841564,6.181559,6.008707,...,25.708821,0.0,1.573794,1.317728,0.0,0.198830,7.975450,9.789609,1.183271,258.838710
min,2026-10-04 00:00:00,5.526038e+07,1.206361e+06,55.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2026-10-05 00:00:00,9.420105e+08,1.212287e+06,1304.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,28.000000
50%,2026-10-07 00:00:00,1.640260e+09,1.214096e+06,1831.000000,0.000000,15.000000,15.000000,5.000000,0.000000,0.000000,...,14.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,90.000000
75%,2026-10-09 00:00:00,2.072260e+09,1.217229e+06,2269.750000,0.000000,75.000000,30.000000,72.000000,5.000000,8.000000,...,35.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,7.000000,0.000000,297.000000
max,2026-10-10 00:00:00,5.288260e+09,1.221358e+06,7098.000000,540.000000,1800.000000,1020.000000,3648.000000,495.000000,263.000000,...,714.000000,0.0,560.000000,758.000000,0.0,157.000000,1080.000000,749.000000,160.000000,6849.000000
std,NaN,6.603706e+08,2.911810e+03,1551.715687,20.678889,148.115990,65.238840,225.792610,20.927307,13.107029,...,42.571959,0.0,14.522057,21.135046,0.0,2.490927,46.328763,31.213995,5.566863,466.726571


In [15]:
invoices.shape

(7006, 24)

In [16]:
credit_notes.head()

,Date,No,Invoice No.,A/C,Customer,P,S,B,200g,O.P.,...,600g,KATIKATI,MANDAZI,DOUGHNUTS,50-50 SLICED,MANDO FAMILY PACK,4 Square,XTRA LOONG,B-BITEZ,Qty
0,2026-10-04,639026246,1212209,6390,EXTRA SALES - OTHERS 4 (6390)KIAMBU SALES,-15,-360,-165,-72,-15,...,-210,0,0,0,0,0,0,0,0,-861
1,2026-10-04,525026068,1212211,5250,MISSIONARIES OF CHARITY - HURUMA (5250),0,-120,-30,0,0,...,0,0,0,0,0,0,0,0,0,-150
2,2026-10-04,897260320,1212212,897,PHINEAS BUNDI KIOGORA (0897),-15,-120,-105,-216,-15,...,-21,0,0,0,0,0,-15,-7,0,-514
3,2026-10-04,1700260391,1212217,1700,MUHINDI MWEUSI SUPERMARKET - KAYOLE (1700),0,-30,-20,0,0,...,-50,0,0,0,0,0,0,0,0,-125
4,2026-10-04,1721260611,1212218,1721,QUICK MART LIMITED - RUIRU (1721),0,-30,0,-20,0,...,-42,0,0,0,0,0,0,0,0,-110


In [17]:
credit_notes.tail()

,Date,No,Invoice No.,A/C,Customer,P,S,B,200g,O.P.,...,600g,KATIKATI,MANDAZI,DOUGHNUTS,50-50 SLICED,MANDO FAMILY PACK,4 Square,XTRA LOONG,B-BITEZ,Qty
1976,2026-10-10,1822260632,1219791,1822,CLEANSHELF SUPERMARKET - NAKURU (1822),0,0,0,0,-1,...,0,0,0,0,0,0,0,0,0,-1
1977,2026-10-10,2244260452,1219792,2244,JAMAA SUPERMARKET - NAKURU (2244),0,0,0,0,-10,...,-1,0,0,0,0,0,0,0,0,-11
1978,2026-10-10,2026260558,1219795,2026,KHETIA DRAPERS LTD - HIGHLANDS ELDORET (2026),0,0,0,0,-1,...,0,0,0,0,0,0,0,0,0,-1
1979,2026-10-10,2026260559,1219796,2026,KHETIA DRAPERS LTD - HIGHLANDS ELDORET (2026),0,0,0,-1,0,...,0,0,0,0,0,0,0,0,0,-1
1980,2026-10-10,1996260719,1219799,1996,NAIVAS SUPERMARKET - ELGON VIEW ELDORET (1996),0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,-1


In [18]:
credit_notes.shape

(1981, 23)

In [19]:
credit_notes.columns

Index(['Date', 'No', 'Invoice No.', 'A/C', 'Customer', 'P', 'S', 'B', '200g',
       'O.P.', '800g', '1500g', 'ST', '600g', 'KATIKATI', 'MANDAZI',
       'DOUGHNUTS', '50-50 SLICED', 'MANDO FAMILY PACK', '4 Square',
       'XTRA LOONG', 'B-BITEZ', 'Qty'],
      dtype='object')

In [20]:
credit_notes.isna().sum()

Date                 0
No                   0
Invoice No.          0
A/C                  0
Customer             0
P                    0
S                    0
B                    0
200g                 0
O.P.                 0
800g                 0
1500g                0
ST                   0
600g                 0
KATIKATI             0
MANDAZI              0
DOUGHNUTS            0
50-50 SLICED         0
MANDO FAMILY PACK    0
4 Square             0
XTRA LOONG           0
B-BITEZ              0
Qty                  0
dtype: int64

In [21]:
credit_notes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1981 entries, 0 to 1980
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Date               1981 non-null   datetime64[ns]
 1   No                 1981 non-null   int64         
 2   Invoice No.        1981 non-null   int64         
 3   A/C                1981 non-null   int64         
 4   Customer           1981 non-null   object        
 5   P                  1981 non-null   int64         
 6   S                  1981 non-null   int64         
 7   B                  1981 non-null   int64         
 8   200g               1981 non-null   int64         
 9   O.P.               1981 non-null   int64         
 10  800g               1981 non-null   int64         
 11  1500g              1981 non-null   int64         
 12  ST                 1981 non-null   int64         
 13  600g               1981 non-null   int64         
 14  KATIKATI

In [22]:
credit_notes.describe()

,Date,No,Invoice No.,A/C,P,S,B,200g,O.P.,800g,...,600g,KATIKATI,MANDAZI,DOUGHNUTS,50-50 SLICED,MANDO FAMILY PACK,4 Square,XTRA LOONG,B-BITEZ,Qty
count,1981,1.981000e+03,1.981000e+03,1981.000000,1981.000000,1981.000000,1981.000000,1981.000000,1981.000000,1981.000000,...,1981.000000,1981.0,1981.000000,1981.000000,1981.0,1981.000000,1981.000000,1981.000000,1981.000000,1981.000000
mean,2026-10-08 05:23:28.379606272,1.629863e+09,1.216887e+06,1825.372539,-0.522463,-10.669359,-4.562342,-8.809187,-0.711257,-1.158506,...,-3.537607,0.0,-0.335689,-0.262998,0.0,-0.037355,-0.678950,-0.653710,-0.407875,-32.347299
min,2026-10-04 00:00:00,5.526038e+07,1.209623e+06,55.000000,-135.000000,-1157.000000,-732.000000,-3144.000000,-164.000000,-42.000000,...,-210.000000,0.0,-160.000000,-150.000000,0.0,-10.000000,-270.000000,-48.000000,-37.000000,-5271.000000
25%,2026-10-08 00:00:00,1.224261e+09,1.215956e+06,1274.000000,0.000000,-1.000000,-1.000000,0.000000,0.000000,-1.000000,...,-2.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,-8.000000
50%,2026-10-08 00:00:00,1.733261e+09,1.216699e+06,1763.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,-4.000000
75%,2026-10-09 00:00:00,2.104260e+09,1.218230e+06,2141.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,-2.000000
max,2026-10-10 00:00:00,2.494260e+09,1.219799e+06,7098.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,-1.000000
std,NaN,5.559518e+08,1.739453e+03,993.887118,5.339213,51.199986,23.430852,99.813204,5.601986,3.097264,...,13.257825,0.0,4.557698,3.518492,0.0,0.456842,8.205622,3.586623,2.222752,178.204934


In [23]:
numeric_columns = ['P', 'S', 'B', '200g', 'O.P.', '800g', '600g', 'MANDAZI', 'DOUGHNUTS', 'MANDO FAMILY PACK', '4 Square', 'XTRA LOONG', 'B-BITEZ', 'Qty']

In [24]:
credit_notes.columns

Index(['Date', 'No', 'Invoice No.', 'A/C', 'Customer', 'P', 'S', 'B', '200g',
       'O.P.', '800g', '1500g', 'ST', '600g', 'KATIKATI', 'MANDAZI',
       'DOUGHNUTS', '50-50 SLICED', 'MANDO FAMILY PACK', '4 Square',
       'XTRA LOONG', 'B-BITEZ', 'Qty'],
      dtype='object')

In [25]:
# Function to check if invoice row matches any credit note row for the same A/C
def is_matching_credit_row(invoice_row, credit_notes, numeric_columns):
    ac = invoice_row['A/C']
    # Filter credit notes for the same A/C
    credit_notes_ac = credit_notes[credit_notes['A/C'] == ac]
    if credit_notes_ac.empty:
        return False
    # Compare numeric columns, accounting for negative values in credit notes
    for _, credit_row in credit_notes_ac.iterrows():
        if all(invoice_row[col] == -credit_row[col] for col in numeric_columns):
            return True
    return False

In [26]:
# Process data datewise and aggregate results (remove only one matching invoice per credit note per day)
all_results = []
if not invoices.empty:
    unique_dates = invoices['Date'].dropna().unique()
    unique_dates = sorted(unique_dates)
    for date in unique_dates:
        # Filter data for the current date
        daily_invoices = invoices[invoices['Date'] == date].copy()
        if not credit_notes.empty:
            daily_credit_notes = credit_notes[credit_notes['Date'] == date].copy()
        else:
            daily_credit_notes = pd.DataFrame()

        # Remove only one matching invoice per credit note for the day
        if not daily_credit_notes.empty:
            daily_invoices = remove_one_invoice_per_credit(daily_invoices, daily_credit_notes, numeric_columns)
        else:
            daily_invoices['is_credit'] = False

        # Filter out matching invoice rows
        filtered_df = daily_invoices[~daily_invoices['is_credit']].copy()
        # Drop the helper column
        filtered_df = filtered_df.drop(columns=['is_credit'])

        # Keep all customer rows for the day (no grouping/summing)
        if not filtered_df.empty:
            all_results.append(filtered_df)

# Combine all daily results, then sum numeric columns per A/C (one row per A/C); keep the first date seen for reference
if all_results:
    combined_df = pd.concat(all_results, ignore_index=True)
    final_df = (
        combined_df
        .sort_values(by=['Date'])
        .groupby('A/C', as_index=False)
        .agg({**{'Date': 'first'}, **{col: 'sum' for col in numeric_columns}})
        .sort_values(by=['A/C'])
    )
else:
    final_df = pd.DataFrame()

# Display the final combined result
final_df

,A/C,Date,P,S,B,200g,O.P.,800g,600g,MANDAZI,DOUGHNUTS,MANDO FAMILY PACK,4 Square,XTRA LOONG,B-BITEZ,Qty
0,55,2026-10-04,0,1845,720,1416,225,0,217,0,0,0,0,147,30,4600
1,57,2026-10-04,615,6165,2655,3888,390,0,1061,0,0,0,1035,189,10,16008
2,62,2026-10-04,90,4725,1770,3744,165,48,2254,0,0,0,75,336,0,13207
3,63,2026-10-04,525,7050,2025,3000,0,66,161,0,0,0,0,910,30,13767
4,82,2026-10-04,240,1440,600,1608,45,18,98,0,0,0,270,42,10,4371
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
887,7085,2026-10-04,0,140,90,35,0,0,155,0,0,0,40,0,0,460
888,7089,2026-10-05,0,60,45,0,0,12,77,0,0,0,0,0,0,194
889,7094,2026-10-06,0,34,29,0,0,19,34,0,0,0,0,0,0,116
890,7096,2026-10-04,0,60,45,0,10,36,49,0,0,0,0,0,0,200


In [27]:
final_df = final_df.to_excel('2026-BRANDS\\FINAL-OUTPUT\\OCTOBER\\OCT-INV-WK-2.xlsx', index=False)